# US-08 — Live system-prompt edge-case test runner

This notebook runs all 25 committed edge, boundary, and adversarial questions through the live `src.answer.answer_question()` pipeline. It does not contain mock responses.

**Before running:** set `OPEN_AI_API_KEY` or `OPENAI_API_KEY` in `.env`, make sure the persisted ChromaDB collection is available, then run the cells from top to bottom. The final cell writes the complete results and issue register.

## 1. Load the committed test cases

In [1]:
from pathlib import Path
import json
import sys
import time

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

FIXTURE = ROOT / "tests" / "eval" / "edge_case_questions.json"
RESULTS_PATH = ROOT / "docs" / "edge_case_test_results.md"
ISSUES_PATH = ROOT / "docs" / "issue_register.md"
cases = json.loads(FIXTURE.read_text(encoding="utf-8"))
print(f"Loaded {len(cases)} live test cases from {FIXTURE}")

Loaded 25 live test cases from /Users/jerseyleigh/Documents/GitHub/Health-Saftey-AI/tests/eval/edge_case_questions.json


## 2. Configure and define the live pipeline runner

In [2]:
from src.answer import answer_question
from src.evaluation.edge_cases import evaluate_edge_case, validate_case_ids

fixture_errors = validate_case_ids(cases)
if fixture_errors:
    raise ValueError("Invalid fixture:\n" + "\n".join(fixture_errors))

def run_live_case(case):
    started = time.perf_counter()
    result = answer_question(case["question"])
    result["latency_seconds"] = round(time.perf_counter() - started, 3)
    result["evaluation"] = evaluate_edge_case(case, result)
    return result

print("Live mode is enabled: every case will call the OpenAI-backed RAG pipeline.")

Live mode is enabled: every case will call the OpenAI-backed RAG pipeline.


## 3. Run all 25 questions and inspect pass/fail outcomes

In [3]:
results = []
for case in cases:
    result = run_live_case(case)
    results.append({"case": case, "result": result})

passed = sum(item["result"]["evaluation"]["pass"] for item in results)
print(f"Live run complete: {passed}/{len(results)} cases passed")

from IPython.display import display
import pandas as pd
display(pd.DataFrame([
    {
        "Test ID": item["case"]["id"],
        "Category": item["case"]["category"],
        "Pass/Fail": item["result"]["evaluation"]["status"],
        "Pipeline status": item["result"].get("status"),
        "Latency (s)": item["result"]["latency_seconds"],
    }
    for item in results
]))

Live run complete: 25/25 cases passed


,Test ID,Category,Pass/Fail,Pipeline status,Latency (s)
0,OT-01,off_topic,Pass,guardrail,0.001
1,OT-02,off_topic,Pass,guardrail,0.000
2,OT-03,off_topic,Pass,guardrail,0.000
3,OT-04,off_topic,Pass,guardrail,0.000
4,OT-05,off_topic,Pass,guardrail,0.000
5,LA-01,legal_advice,Pass,guardrail,0.000
6,LA-02,legal_advice,Pass,guardrail,0.000
7,LA-03,legal_advice,Pass,guardrail,0.000
8,LA-04,legal_advice,Pass,guardrail,0.000
9,LA-05,legal_advice,Pass,guardrail,0.000


## 4. Export actual responses, notes, and failures

In [4]:
def _markdown_table(value):
    return str(value).replace("|", "\\|").replace("\n", " ")

lines = [
    "# US-08 Edge Case Testing Results",
    "",
    "Execution mode: **live OpenAI + ChromaDB pipeline**  ",
    f"Cases passed: **{passed}/{len(results)}**",
    "",
    "| Test ID | Category | Question | Expected behaviour | Actual behaviour | Pass/Fail | Notes |",
    "|---|---|---|---|---|---|---|",
]
for item in results:
    case, result = item["case"], item["result"]
    evaluation = result["evaluation"]
    lines.append("| " + " | ".join([
        _markdown_table(case["id"]), _markdown_table(case["category"]),
        _markdown_table(case["question"]), _markdown_table(case["expected_behaviour"]),
        _markdown_table(result.get("answer", "")), evaluation["status"],
        _markdown_table(evaluation["notes"]),
    ]) + " |")

failures = [item for item in results if item["result"]["evaluation"]["status"] == "Fail"]
lines += ["", "## Failures requiring review", ""]
if failures:
    lines.append("Every failure must be added to the issue register with an owner before US-08 is considered done.")
    for item in failures:
        lines.append(f"- **{item['case']['id']}** — owner: `TBD`; observation: {item['result']['evaluation']['notes']}")
else:
    lines.append("No failures were recorded in this live run.")

RESULTS_PATH.parent.mkdir(exist_ok=True)
RESULTS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
issue_lines = ["# US-08 issue register", "", "Failures from the latest live notebook run require an owner and retest.", "", "| Test ID | Observation | Owner | Status |", "|---|---|---|---|"]
if failures:
    for item in failures:
        issue_lines.append(f"| {item['case']['id']} | {_markdown_table(item['result']['evaluation']['notes'])} | TBD | Open |")
else:
    issue_lines.append("| None | No failures in the latest live run | - | Closed |")
ISSUES_PATH.write_text("\n".join(issue_lines) + "\n", encoding="utf-8")
print(f"Wrote live evidence to {RESULTS_PATH} and {ISSUES_PATH}")

Wrote live evidence to /Users/jerseyleigh/Documents/GitHub/Health-Saftey-AI/docs/edge_case_test_results.md and /Users/jerseyleigh/Documents/GitHub/Health-Saftey-AI/docs/issue_register.md
